<img src="../Images/DSC_Logo.png" style="width: 400px;">

# Basics of Quantitative Data Analysis in Python
# - Loading Files & Data Structures

Research often starts with existing data files: for example tables from surveys, lab measurements, sensor time series, administrative records, or model outputs. Before data can be analyzed, it first has to be loaded correctly and understood in a suitable structure.

This notebook introduces the basic ways to work with quantitative data in Python. We focus on:
- loading common file types such as **text files** for simple file input/output
- understanding key data structures for quantitative work: **dataframes** with `pandas` for tabular data and **arrays** with `numpy` for numerical data
- inspecting and selecting data
- converting between structures when needed

The main goal is not yet to analyze data, but to learn how to bring data into Python in a structured and reproducible way.

In [ ]:
# Install all required libraries for this notebook:

!pip install -q -r ../requirements_B.txt   

print("The libraries have been installed.")

## 1. Reading and Writing Text Files

In research, data is often stored in files rather than created directly in code. While many quantitative datasets are later loaded with libraries such as `pandas`, it is still useful to understand the basic Python way of opening and writing files.

Python provides the built-in function `open()` for this purpose. It is commonly used together with a `with` statement, which automatically closes the file again after use.

Let's **open** a file (from a relative path) in `read` mode and store its contents in the variable `data`:

In [ ]:
with open("../Data/DataB/Iris.csv", "r") as f: 
    data = f.read()

print(data)

To **save** data into a new file, we change the mode to `write`:

In [ ]:
new_string = "These are my notes on experiment A."

with open("../Data/DataB/Notes.txt", "w") as f:
    f.write(new_string)

However, this way of reading and writing files does not interpret comma-separated values automatically. If you want to work with structured tabular data such as CSV files, it is better to use libraries such as `csv` or `pandas` (Sect. 2).

## 2. File Organization and Batch Processing

In many research projects, data is stored across multiple files and folders. Instead of opening each file manually, it is often more efficient and less error-prone to work with paths programmatically and **process groups of files** automatically.

Python’s `pathlib` library helps define files and folders clearly and makes code for file handling easier to read.






In [ ]:
from pathlib import Path    # Import library (Python built-in)

Define the "Samples" folder:

In [ ]:
sample_dir = Path("../Data/DataB/Samples")  

The `glob` method can be used to search for files that match a specific pattern. In the example below, we use `glob` to search for all .txt files in the "Samples" folder:

In [ ]:
files = sample_dir.glob("*.txt")

?:

In [ ]:
for filepath in files:
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
        print(content)

---
### **Exercise 1:**  
Please answer the following questions about the code above. Write your answers in **Markdown** in the `:?` cell above the code.

- Which basic Python structure is used here to repeat steps several times?
- What happens in the code each time the steps are repeated?
- How could you make the code repeat more times?

Solution: 
- The code uses a **`for` loop**.
- It goes through all text files in a folder, opens each file, reads its content, and prints the content to the screen.
- The code repeats more times if `files` contains more file paths, for example because more matching text files are placed in the folder.

---

## 3. Tabular Data / `pandas`

## 3.1 Basics

Because many quantitative datasets can be represented as tables, the **dataframe** is often the most important structure to learn first. Datasets such as survey results, administrative records, lab measurements, or outputs from instruments and models are often stored in a tabular format, with rows representing observations and columns representing variables.

The Python library `pandas` is designed for working with this kind of tabular data and is the most widely used tool for it in Python. `pandas` reads files such as .csv or Excel spreadsheets into dataframes, which keep both the data and its structure intact.

<img src="../Images/dataframe.png" style="width: 300px;">

In [ ]:
import pandas as pd     # Import library (must be installed)

To **create** a dataframe (`DataFrame`) from a dictionary:

In [ ]:
names_dict = {"numbers": [1, 2, 3, 4, 5, 6]}
df = pd.DataFrame(names_dict)

print(df)

To **save** a dataframe to CSV:

In [ ]:
df.to_csv("../Data/DataB/numbers.csv", index=False)     # Setting `index=False` prevents pandas from writing row indices to the file.

When **importing** tabular data, common adjustments include:
- choosing a separator, e.g. `sep=";"` instead of comma
- skipping metadata rows with `skiprows=...`
- specifying missing-value markers with `na_values=...`
- reading a specific Excel sheet with `sheet_name=...`

For the text file example from Section 1, none of these import adjustments are needed. In practice, however, it is always important to check such settings in advance.

In [ ]:
Iris = pd.read_csv("../Data/DataB/Iris.csv")

print(Iris)

When loading a new dataset, the first step is usually not analysis, but inspection: checking dimensions, column names, data types, and a few example rows. Once the data are in a dataframe, we can check:

- How many rows and columns do we have?

In [ ]:
print(Iris.shape)   # (number_of_rows, number_of_columns)

- What columns exist in this dataframe?

In [ ]:
print(Iris.columns)

- 10 randomly selected rows:

In [ ]:
Iris.sample(10)

A dataframe follows Python’s usual indexing logic, so we can, for example, **select** a specific column either by its name or its position (index).

- Select a column by name:

In [ ]:
species = Iris["Species"]

print(species)

- Select the first column by its index position (column 0):

In [ ]:
first_column = Iris.iloc[:, 0]

print(first_column)

- Select the first row by its index position (row 0):

In [ ]:
first_row = Iris.iloc[0, :]

print(first_row)

We can also **add data** to an existing dataframe. 

- Add a new empty column with missing values:

In [ ]:
Iris["Notes"] = pd.NA

print(Iris)

- Fill a few rows:

In [ ]:
Iris.loc[0:2, "Notes"] = "check"

Iris.head()      # .head() displays only the first five rows; also check out .tail() and .sample()

---

### **Exercise 2:** 

`pandas` can also be used to import other tabular file formats, including **Excel files**. Instead of `pd.read_csv()`, use `pd.read_excel()` to import the file `World-happiness-report-2024.xlsx` as a dataframe. Then print all column names and select one column.

In [ ]:
# Solution:

# Load the file as a pandas dataframe
report = pd.read_excel("../Data/DataB/World-happiness-report-2024.xlsx") 

# Print all column names
print(report.columns)

# Select one column
country_name = Iris.iloc[:, 0]
print(country_name)

---

There are several ways to **put tabular data together** in `pandas`. For example, `concat()` can stack dataframes, while `merge()` combines them based on shared keys or IDs.  

Below, we see a simple example with `merge()`. For more options, see the pandas documentation: [Merge, join, concatenate and compare](https://pandas.pydata.org/docs/user_guide/merging.html)

In [ ]:
# Create first dataframe
scores = pd.DataFrame({
    "ID": [1, 2, 3],
    "Score": [12, 15, 14]
})

# Create second dataframe
groups = pd.DataFrame({
    "ID": [1, 2, 3],
    "Group": ["A", "B", "A"]
})

# Merge both dataframes using the shared ID column
merged_data = pd.merge(scores, groups, on="ID")  # <-
print(merged_data)

## 3.2 Wide and Long Data Format

Tabular data can be organized in different ways. Two common formats are wide and long:
- In **wide format**, repeated measurements are spread across several columns.
- In **long format**, each row represents one observation at one measurement occasion, and repeated measurements are stored in a single measurement column together with an additional column indicating time, condition, or category.

Many plotting, grouping, and modeling functions in Python work especially well with long-format data.

Below, we create a dataframe where each participant of a survey appears only once, and the repeated measurements are stored in separate columns (wide format):

In [ ]:
wide_data = pd.DataFrame({
    "participant_id": ["P01", "P02", "P03"],
    "score_t1": [12, 15, 14],
    "score_t2": [13, 16, 15]
})

print(wide_data)

The function `.melt()` **reshapes** the table from wide format to long format. In the long format, each participant can appear multiple times, and the repeated measurements are stored in separate rows (not separate columns).

In [ ]:
long_data = wide_data.melt(
    id_vars="participant_id",
    var_name="time",
    value_name="score"
)

print(long_data)

Reshape long format back to wide format with `.pivot()`:

In [ ]:
wide_data = long_data.pivot(
    index="participant_id",
    columns="time",
    values="score"
)

print(wide_data)

## 4. Data Arrays / `numpy`

While `pandas` dataframes are ideal for tabular datasets with named columns, many scientific applications also use **arrays**. These are especially useful when data has a regular numerical structure, such as vectors, matrices, images, or gridded values.

The `numpy` library provides the core array structure used throughout Python. Many other Python libraries are built on top of `numpy`, including `pandas`. 

Arrays can be one-dimensional, two-dimensional, or have even more dimensions. The **shape of an array** is very important. Many operations only work when array shapes are compatible. The exact requirement depends on the task.

<img src="../Images/array.png" style="width: 600px;"> <img src="../Images/pandas_numpy.png" style="width: 300px;">

In [ ]:
import numpy as np      # Import library (must be installed)

Let's **import** some data (temperature anomaly time series) as a 2D NumPy array:

In [ ]:
time_series = np.loadtxt('../Data/DataB/NOAA_time_series.csv', skiprows=5, delimiter=',')

print(time_series) 

Let's see how to inspect the **structure** of an array!

- Size of each dimension:

In [ ]:
print(time_series.shape)

- Number of dimensions:

In [ ]:
print(time_series.ndim)

- Data type of the stored values:

In [ ]:
print(time_series.dtype)

To **create** a 2D NumPy array from scratch:

In [ ]:
data = np.array([
    [1, 0, 0, 1],
    [1, 0, 1, 1],
    [0, 1, 1, 1],
    [0, 0, 0, 1]
])

print(data.shape)

Now we import `matplotlib` to quickly **visualize** our NumPy array and make its structure (rows, columns, and patterns) easier to understand.

In [ ]:
import matplotlib.pyplot as plt     # Import library (must be installed)

In [ ]:
plt.imshow(data, plt.cm.gray_r)

Below, we aim to **select** only parts of the array/image.

- First element/cell:

In [ ]:
print(data[0, 0])

- First two rows and first two columns:

In [ ]:
print(data[0:2,0:2])

There are also several ways to **manipulate and combine arrays** in `numpy`, for example by adding, stacking, reshaping, or concatenating data.  

Below is one simple example for adding a new row to the `data` array, but many other options are available in the NumPy documentation: [Array manipulation routines](https://numpy.org/doc/stable/reference/routines.array-manipulation.html#).

In [ ]:
# Create the new row as a NumPy array
new_row = np.array([[1, 1, 1, 1]])

# Stack the arrays vertically to add the new row
data = np.vstack([data, new_row])

print(data)

NumPy can also handle **multi-dimensional data**, such as 3D arrays, which are useful for representing things like image stacks or time-varying data. Here's a basic example for creating a 3D array: 2 "layers", each 3x3:

In [ ]:
data = np.array([
    [[1, 2, 3],
     [4, 5, 6],
     [7, 8, 9]],

    [[10, 11, 12],
     [13, 14, 15],
     [16, 17, 18]]
])

print(data.shape)

---
### **Exercise 3:** 

Create a small 2D NumPy array (e.g., 9×9) filled with zeros, then turn some cells into 1s so that `plt.imshow()` shows a circle/ring.

In [ ]:
# Solution:

# Create 2D NumPy array
data = np.array([
    [0,0,0,1,1,1,0,0,0],
    [0,0,1,0,0,0,1,0,0],
    [0,1,0,0,0,0,0,1,0],
    [1,0,0,0,0,0,0,0,1],
    [1,0,0,0,0,0,0,0,1],
    [1,0,0,0,0,0,0,0,1],
    [0,1,0,0,0,0,0,1,0],
    [0,0,1,0,0,0,1,0,0],
    [0,0,0,1,1,1,0,0,0],
])

# Plot with matplotlib
plt.imshow(data, plt.cm.gray_r)

---

`numpy` arrays are great for fast numerical operations. However, they lack labels (like column names), which makes dataframes more convenient for data exploration and analysis.

Here is how you can **convert** between the two data structures:

- Convert array to dataframe:

In [ ]:
df = pd.DataFrame(time_series, columns=['Year', 'Anomaly'])

df.head()       # .head prints the first 5 rows of the df by default

- Convert dataframe to array:

In [ ]:
array_again = df.values
print(array_again[:5]) 

## 5. More Data Structures: Labelled Multidimensional Data / `xarray`

Some scientific datasets are not best represented as simple tables or unlabeled arrays. For example, climate, remote sensing, or model data often has several labelled dimensions such as time, latitude, and longitude.

The library `xarray` is designed for this type of data. Below, we only take a brief look at the basic idea.

In [ ]:
import xarray as xr     # Import library (must be installed)

When working with `xarray`, we distinguish between:
- **dimensions:** the axes along which the data varies, for example time, latitude, and longitude
- **coordinates:** the labels along these dimensions, for example specific dates or latitude values
- **variables:** the actual measured values, for example temperature or precipitation

To illustrate the idea, we can first create a very small labelled data structure from scratch:

In [ ]:
temperature = xr.DataArray(
    np.random.rand(3, 4),
    dims=("time", "location"),
    coords={
        "time": ["day1", "day2", "day3"],
        "location": ["A", "B", "C", "D"]
    }
)

print(temperature)

Here, the data has two dimensions (`time` and `location`) and labelled coordinates along both axes. This is similar to an array in `numpy`, but with additional labels that make the structure easier to interpret.

Such labelled multidimensional data is often stored in specialized file formats. A common format is **NetCDF**, which is widely used for climate, and other gridded scientific datasets. As an example, we use a small snippet of ERA5 climate reanalysis data, which contains near-surface air temperature values (`t2m`) on a spatial grid over time.

If not unpacked already, the .7z archive containing the NetCDF file first needs to be extracted:

In [ ]:
import py7zr        # Import libraries (must be installed)

In [ ]:
with py7zr.SevenZipFile('../Data/DataB/ERA5_snippet.7z', mode='r', password='secret') as archive:
    archive.extractall(path='../Data/DataB/')

**Import** the file:

In [ ]:
ERA5 = xr.open_dataset('../Data/DataB/ERA5_snippet.nc')
print(ERA5)

Let's inspect the **structure** of the dataset:

- Show the dimensions of the dataset:

In [ ]:
print(ERA5.dims)

- Show the coordinate variables:

In [ ]:
print(ERA5.coords)

- Show the data variables stored in the dataset:

In [ ]:
print(ERA5.data_vars)

Let's see how to **select** data in `xarray` multidimensional datasets.

- First, we select t2m, which represents 2 meter air temperature. Selecting a single variable from an `xarray` dataset works similarly to selecting a single column from a `pandas` dataframe:

In [ ]:
t2m = ERA5["t2m"]

print(t2m)

- We can also select along dimensions. To select the first time step by index with `.isel()`:

In [ ]:
t2m.isel(time=0)

- To select a single value by index position with .isel():

In [ ]:
t2m.isel(time=0, latitude=0, longitude=0)

- To select data by coordinate labels:

In [ ]:
t2m.sel(time="2023-01-01")

We **plot** the first time slice of temperature (t2m) to get a better understanding of the data structure:

In [ ]:
t2m.isel(time=0).plot()

To **convert** to a `pandas` dataframe:

In [ ]:
t2m_df = t2m.to_dataframe().reset_index()

t2m_df.head()

## Key Takeaways

In this notebook, you learned that:

- Python can load data from files using basic file handling or specialized libraries.
- `pandas` dataframes are the main structure for tabular quantitative data.
- `numpy` arrays are useful for regular numerical data such as vectors, matrices, or images.
- Beyond dataframes and arrays, Python also provides specialized libraries for other data structures. `xarray` is one example for labelled multidimensional data.
- Before analysis, it is important to inspect the structure of loaded data carefully.